In [0]:
%sql
SELECT 
    s.s_store_name,

    c.c_customer_id,
    c.c_first_name || ' ' || c.c_last_name AS customer_name,
    c.c_preferred_cust_flag,
    c.c_birth_country,
    COUNT(DISTINCT ss.ss_ticket_number) AS total_purchases,
    SUM(ss.ss_quantity) AS total_items_purchased,
    ROUND(SUM(ss.ss_net_paid), 2) AS total_spent,
    ROUND(AVG(ss.ss_net_paid), 2) AS avg_purchase_value,
    ROUND(SUM(ss.ss_net_profit), 2) AS profit_generated,
    MIN(d.d_date) AS first_purchase_date,
    MAX(d.d_date) AS last_purchase_date,

    RANK() OVER (PARTITION BY s.s_store_sk ORDER BY SUM(ss.ss_net_paid) DESC) AS customer_rank_in_store
FROM 
    samples.tpcds_sf1.store_sales ss
    INNER JOIN samples.tpcds_sf1.customer c ON ss.ss_customer_sk = c.c_customer_sk
    INNER JOIN samples.tpcds_sf1.store s ON ss.ss_store_sk = s.s_store_sk
    INNER JOIN samples.tpcds_sf1.date_dim d ON ss.ss_sold_date_sk = d.d_date_sk
WHERE 
    s.s_rec_end_date IS NULL  -- only open stores
    AND d.d_year >= 1998
GROUP BY 
    s.s_store_sk,
    s.s_store_name,

    c.c_customer_sk,
    c.c_customer_id,
    c.c_first_name,
    c.c_last_name,
    c.c_preferred_cust_flag,
    c.c_birth_country
HAVING 
    COUNT(DISTINCT ss.ss_ticket_number) >= 5  -- due to the sample data 5 purchases is the max amount a customer has
ORDER BY 
    s.s_store_name,
    total_spent DESC

In [0]:
%sql
SELECT 
    s.s_store_name,
    s.s_store_id,
    d.d_year,
    d.d_qoy AS quarter,
    COUNT(DISTINCT ss.ss_customer_sk) AS total_customers,
    COUNT(ss.ss_ticket_number) AS total_transactions,
    SUM(ss.ss_quantity) AS total_items_sold,
    ROUND(SUM(ss.ss_net_paid), 2) AS total_revenue,
    ROUND(SUM(ss.ss_net_profit), 2) AS total_profit,
    ROUND(AVG(ss.ss_net_paid), 2) AS avg_ticket_value,

    RANK() OVER (PARTITION BY d.d_year, d.d_qoy ORDER BY SUM(ss.ss_net_paid) DESC) AS revenue_rank_by_quarter
FROM 
    samples.tpcds_sf1.store_sales ss
    INNER JOIN samples.tpcds_sf1.date_dim d ON ss.ss_sold_date_sk = d.d_date_sk
    INNER JOIN samples.tpcds_sf1.store s ON ss.ss_store_sk = s.s_store_sk
WHERE 
    d.d_year >= 1998
    AND s.s_rec_end_date IS NULL  -- only open stores
GROUP BY 
    s.s_store_name,
    s.s_store_id,
    s.s_store_sk,
    d.d_year,
    d.d_qoy
HAVING 
    SUM(ss.ss_net_paid) > 0
ORDER BY 
    d.d_year DESC,
    d.d_qoy DESC,
    total_revenue DESC

In [0]:
%sql
UPDATE samples.tpcds_sf1.customer c
SET c_preferred_cust_flag = 'Y'
WHERE 
    c.c_preferred_cust_flag = 'N'
    AND c.c_customer_sk IN (
        SELECT ss.ss_customer_sk
        FROM samples.tpcds_sf1.store_sales ss
        INNER JOIN samples.tpcds_sf1.date_dim d ON ss.ss_sold_date_sk = d.d_date_sk
        WHERE d.d_year >= 1998
        GROUP BY ss.ss_customer_sk
        HAVING 
            COUNT(DISTINCT ss.ss_ticket_number) >= 10  -- 
            AND SUM(ss.ss_net_paid) >= 10000  -- 
    );